# TEST ML

## LOAD DATA

In [2]:
import os
import pandas as pd
import duckdb
from functions import choose_features_target
from sklearn.model_selection import TimeSeriesSplit

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_cleaned.parquet')
            """)

df_features = con.execute("SELECT * FROM train_delay").fetchdf()

## QUESTIONS
- how to treat categorical features
- export historical results

## SELECT VARIABLES

In [14]:
df_features.columns.values.tolist()

['ride_id',
 'train_name',
 'train_type',
 'station_current',
 'station_dest',
 'departure_planned',
 'departure_real',
 'arrival_planned',
 'arrival_real',
 'temperature',
 'precipitation',
 'wind_speed',
 'delay',
 'station_start',
 'stops_total',
 'departure_planned_start',
 'arrival_planned_dest',
 'travel_time',
 'weekday',
 'month',
 'feast',
 'hour',
 'dwell_time_planned',
 'station_role',
 'stop_index',
 'time_since_start_planned',
 'share_ride_time',
 'delay_cat']

In [ ]:
# separate features and target
X, y = choose_features_target(df_features)

## TRANSFORMING AND PREPROCESSING

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder


# ONE-HOT
# station_current, train_name, train_type, station_start, station_dest, station_role, feast


# SCALER (Standard)
# temperature, precipitation_log, wind_speed, hour_sin, hour_cos, weekday_sin, weekday_cos
# travel_time, dwell_time_planned, stop_index, time_since_start_planned, share_ride_time

# KEIN ENCODING
# precipitation_any, feast, weekend, rush_morning, rush_evening

feature_schema = {
    "time": ["hour", "weekday", "season"],
    "categorical": ["station_current", "train_name", "train_type", "station_start", "station_dest", 
                    "weekend", "feast", "station_role"],
    "numeric": ["travel_time", "dwell_time_planned", "stop_index", "stops_remaining",
                "time_since_start_planned", "time_to_dest_planned", "share_ride_time_completed", "share_ride_stations",
                "hist_delay_station_avg", "hist_delay_station_std", "hist_delay_station_q90", "hist_delay_station_count",
                "hist_delay_station_hour_avg", "hist_delay_station_hour_std", "hist_delay_station_hour_q90", "hist_delay_station_hour_count",
                "hist_delay_station_day_avg", "hist_delay_station_day_std", "hist_delay_station_day_q90", "hist_delay_station_day_count"],
}

### ENCODE CATEGORICAL FEATURES ###
one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", one_hot_encoder, feature_schema["categorical"]),
        ("time", one_hot_encoder, feature_schema["time"]),
        ("num", MinMaxScaler(), feature_schema["numeric"]),
    ]
)


## MODEL

In [5]:
### CHOOSE MODEL ###

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models_list = {
    "linear": LinearRegression(),
    
    "rf": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=50,  
        n_jobs=-1,
        random_state=42,
    ),
    
    "gb": GradientBoostingRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
    ),
}

## PIPELINE


In [18]:
from sklearn.pipeline import make_pipeline

pipeline_first = make_pipeline(
    preprocessor,
    models_list["rf"],
)

## CROSS-VALIDATION

In [7]:
# define time series cross-validator
ts_cv = TimeSeriesSplit(
    n_splits=5,
    gap=48, # should i add a gap?
    max_train_size=10000,
)

# get overview about splitting
for i, (train_index, test_index) in enumerate(ts_cv.split(df["stops_total"])):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")


# check if splitting worked as expected
all_splits = list(ts_cv.split(df))
train_0, test_0 = all_splits[0]
df.iloc[test_0]

Fold 0:
  Train: index=[   0    1    2 ... 2371 2372 2373]
  Test:  index=[2422 2423 2424 ... 4841 4842 4843]
Fold 1:
  Train: index=[   0    1    2 ... 4793 4794 4795]
  Test:  index=[4844 4845 4846 ... 7263 7264 7265]
Fold 2:
  Train: index=[   0    1    2 ... 7215 7216 7217]
  Test:  index=[7266 7267 7268 ... 9685 9686 9687]
Fold 3:
  Train: index=[   0    1    2 ... 9637 9638 9639]
  Test:  index=[ 9688  9689  9690 ... 12107 12108 12109]
Fold 4:
  Train: index=[ 2062  2063  2064 ... 12059 12060 12061]
  Test:  index=[12110 12111 12112 ... 14529 14530 14531]


,id,journey_id,train_name,train_type,station_current,station_dest,time_current_planned,time_current_real,time_current_arrival_planned,delay,...,hist_delay_station_day_std,hist_delay_station_day_q90,hist_delay_station_day_count,station_role,stop_index,stops_remaining,time_since_start_planned,time_to_dest_planned,share_ride_time_completed,share_ride_stations
5137,1090558,305111,ICE 1148,ICE,Wolfsburg Hbf,Frankfurt(Main)Hbf,2025-11-15 10:19:00,2025-11-15 10:23:00,2025-11-15 10:13:00,4,...,21.737243,32.0,5445.0,mid,4,5,117.0,183.0,0.390000,0.444444
5138,1090559,305111,ICE 1148,ICE,Hannover Hbf,Frankfurt(Main)Hbf,2025-11-15 11:17:00,2025-11-15 11:39:00,2025-11-15 11:14:00,22,...,19.253938,29.0,14467.0,mid,5,4,175.0,125.0,0.583333,0.555556
5139,1090560,305111,ICE 1148,ICE,Göttingen,Frankfurt(Main)Hbf,2025-11-15 12:06:00,2025-11-15 12:47:00,2025-11-15 11:55:00,41,...,19.104594,29.0,8907.0,mid,6,3,224.0,76.0,0.746667,0.666667
5140,1090561,305111,ICE 1148,ICE,Kassel-Wilhelmshöhe,Frankfurt(Main)Hbf,2025-11-15 12:28:00,2025-11-15 13:07:00,2025-11-15 12:26:00,39,...,18.820246,28.0,9202.0,mid,7,2,246.0,54.0,0.820000,0.777778
5141,1090562,305111,ICE 1148,ICE,Marburg (Lahn),Frankfurt(Main)Hbf,2025-11-15 13:22:00,2025-11-15 14:00:00,2025-11-15 13:20:00,38,...,21.275074,13.0,12.0,mid,8,1,300.0,0.0,1.000000,0.888889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2130,450089,132553,IC 2207,IC,Lingen (Ems),Köln Hbf,2025-12-13 17:44:00,2025-12-13 17:46:00,2025-12-13 17:42:00,2,...,13.369414,23.0,57.0,mid,5,9,71.0,125.0,0.362245,0.357143
13092,3040899,689473,ICE 8,ICE,Bonn Hbf,Hamburg-Altona,2025-12-13 16:45:00,2025-12-13 17:47:00,2025-12-13 16:43:00,62,...,20.660478,40.0,1636.0,mid,7,13,221.0,276.0,0.444668,0.350000
6557,1357687,365908,ICE 1617,ICE,Hamburg Hbf,Berlin Hbf,2025-12-13 17:48:00,2025-12-13 17:48:00,2025-12-13 17:45:00,0,...,18.871771,26.0,11575.0,mid,2,5,13.0,140.0,0.084967,0.285714
55,17206,6255,EC 172,EC,Dresden Hbf,Berlin Hbf,2025-12-13 16:55:00,2025-12-13 17:50:00,2025-12-13 16:50:00,55,...,15.959477,14.0,2803.0,mid,2,3,30.0,143.0,0.173410,0.400000


## TRAINING AND EVALUATION

In [8]:
### EVALUATE MODEL ###
from sklearn.model_selection import cross_validate

def evaluate(pipeline, X, y, cv):
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
        },
        n_jobs=-1,
    )

    mae = -scores["test_mae"]
    rmse = -scores["test_rmse"]

    return {
    "MAE_mean": float(mae.mean()),
    "MAE_std": float(mae.std()),
    "RMSE_mean": float(rmse.mean()),
    "RMSE_std": float(rmse.std()),
    }



In [22]:
evaluate(pipeline_first, X, y, cv=ts_cv)

{'MAE_mean': 5.33133099936124,
 'MAE_std': 1.3522332903433836,
 'RMSE_mean': 8.50027479576508,
 'RMSE_std': 1.4603419355195666}

## SAVE MODEL

In [22]:
import joblib

# Suppose you already fitted your pipeline
pipeline_first.fit(X, y["delay"])

# Export to a file
joblib.dump(pipeline_first, "trained_delay_model.pkl")
print("Pipeline saved as 'trained_delay_model.pkl'")


Pipeline saved as 'trained_delay_model.pkl'
